In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("GPU not detected. Check Runtime settings.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.3 GB
BF16 supported: True


In [ ]:
%pip install -q \
    "transformers==4.46.3" \
    "datasets==2.21.0" \
    "safetensors==0.5.3" \
    "accelerate"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires safetensors>=0.8.0, but you have safetensors 0.5.3 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

# Keep the model cache on Colab's local disk for faster access.
os.environ["HF_HOME"] = "/content/hf_cache"

# Save completed embeddings persistently on Google Drive.
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/LoRe/PersonalLLM_embeddings"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Directory exists:", OUTPUT_DIR.exists())

Mounted at /content/drive
Output directory: /content/drive/MyDrive/LoRe/PersonalLLM_embeddings
Directory exists: True


In [ ]:
from datasets import load_dataset

dataset = load_dataset("namkoong-lab/PersonalLLM")

print(dataset)
print()

for split in dataset:
    print(f"{split}: {len(dataset[split]):,} prompts")
    print("Columns:", dataset[split].column_names)
    print()

row = dataset["train"][0]

print("Prompt example:")
print(str(row["prompt"])[:500])
print("\nResponse fields:")
print([key for key in row if key.startswith("response_")][:12])

Generating train split:   0%|          | 0/9402 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'subset', 'prompt_id', 'response_1', 'response_1_model', 'response_2', 'response_2_model', 'response_3', 'response_3_model', 'response_4', 'response_4_model', 'response_5', 'response_5_model', 'response_6', 'response_6_model', 'response_7', 'response_7_model', 'response_8', 'response_8_model', 'response_1_gemma_2b', 'response_2_gemma_2b', 'response_3_gemma_2b', 'response_4_gemma_2b', 'response_5_gemma_2b', 'response_6_gemma_2b', 'response_7_gemma_2b', 'response_8_gemma_2b', 'response_1_gemma_7b', 'response_2_gemma_7b', 'response_3_gemma_7b', 'response_4_gemma_7b', 'response_5_gemma_7b', 'response_6_gemma_7b', 'response_7_gemma_7b', 'response_8_gemma_7b', 'response_1_mistral_raft', 'response_2_mistral_raft', 'response_3_mistral_raft', 'response_4_mistral_raft', 'response_5_mistral_raft', 'response_6_mistral_raft', 'response_7_mistral_raft', 'response_8_mistral_raft', 'response_1_mistral_ray', 'response_2_mistral_ray', 'resp

In [ ]:
# Inspect the dataset without downloading it again
print(dataset)

print("\nColumn names:")
print(dataset["train"].column_names)

row = dataset["train"][0]

print("\nPrompt example:")
print(str(row["prompt"])[:500])

print("\nCandidate responses:")
for i in range(1, 9):
    key = f"response_{i}"
    print(f"\n{key}:")
    print(str(row[key])[:250])

print("\nReward-score columns:")
reward_cols = [
    c for c in dataset["train"].column_names
    if c.startswith("response_")
    and c not in [f"response_{i}" for i in range(1, 9)]
]
print(reward_cols)
print("Number of reward-score columns:", len(reward_cols))

DatasetDict({
    train: Dataset({
        features: ['prompt', 'subset', 'prompt_id', 'response_1', 'response_1_model', 'response_2', 'response_2_model', 'response_3', 'response_3_model', 'response_4', 'response_4_model', 'response_5', 'response_5_model', 'response_6', 'response_6_model', 'response_7', 'response_7_model', 'response_8', 'response_8_model', 'response_1_gemma_2b', 'response_2_gemma_2b', 'response_3_gemma_2b', 'response_4_gemma_2b', 'response_5_gemma_2b', 'response_6_gemma_2b', 'response_7_gemma_2b', 'response_8_gemma_2b', 'response_1_gemma_7b', 'response_2_gemma_7b', 'response_3_gemma_7b', 'response_4_gemma_7b', 'response_5_gemma_7b', 'response_6_gemma_7b', 'response_7_gemma_7b', 'response_8_gemma_7b', 'response_1_mistral_raft', 'response_2_mistral_raft', 'response_3_mistral_raft', 'response_4_mistral_raft', 'response_5_mistral_raft', 'response_6_mistral_raft', 'response_7_mistral_raft', 'response_8_mistral_raft', 'response_1_mistral_ray', 'response_2_mistral_ray', 'resp

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "Skywork/Skywork-Reward-Llama-3.1-8B-v0.2"

assert torch.cuda.is_available(), "Please enable a GPU first."

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

model.eval()

print("Model loaded:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Model dtype:", next(model.parameters()).dtype)
print("Model device:", next(model.parameters()).device)
print("Hidden size:", model.config.hidden_size)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/117M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded: Skywork/Skywork-Reward-Llama-3.1-8B-v0.2
GPU: NVIDIA A100-SXM4-80GB
Model dtype: torch.bfloat16
Model device: cuda:0
Hidden size: 4096


In [ ]:
import torch
from pathlib import Path

# Use the first training prompt and its first response.
row = dataset["train"][0]

messages = [
    {"role": "user", "content": row["prompt"]},
    {"role": "assistant", "content": row["response_1"]},
]

# Match LoRe's chat-template tokenization.
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
).to(next(model.parameters()).device)

# Extract the final token's hidden representation.
with torch.inference_mode():
    output = model(input_ids)
    embedding = (
        output.last_hidden_state[0, -1]
        .float()
        .cpu()
        .contiguous()
    )

print("Prompt ID:", row["prompt_id"])
print("Number of tokens:", input_ids.shape[-1])
print("Embedding shape:", tuple(embedding.shape))
print("All finite:", torch.isfinite(embedding).all().item())
print("Embedding norm:", embedding.norm().item())

assert embedding.shape == (4096,)
assert torch.isfinite(embedding).all()

# Persist the test embedding on Google Drive.
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/LoRe/PersonalLLM_embeddings"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "prompt_id": row["prompt_id"],
        "response_index": 1,
        "embedding": embedding,
    },
    OUTPUT_DIR / "smoke_test.pt",
)

print("Saved:", OUTPUT_DIR / "smoke_test.pt")

Prompt ID: 19032
Number of tokens: 278
Embedding shape: (4096,)
All finite: True
Embedding norm: 144.17799377441406
Saved: /content/drive/MyDrive/LoRe/PersonalLLM_embeddings/smoke_test.pt


In [ ]:
import torch
from pathlib import Path

device = next(model.parameters()).device
row = dataset["train"][0]

# Build four prompt-response conversations.
conversations = [
    [
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row[f"response_{j}"]},
    ]
    for j in range(1, 5)
]

# Tokenize each conversation exactly as in the smoke test.
sequences = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_tensors="pt",
    )[0]
    for messages in conversations
]

# Right-pad the sequences and create an attention mask.
pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id

lengths = torch.tensor([len(seq) for seq in sequences])
max_len = int(lengths.max())

input_ids = torch.full(
    (len(sequences), max_len),
    pad_id,
    dtype=torch.long,
)
attention_mask = torch.zeros_like(input_ids)

for i, seq in enumerate(sequences):
    input_ids[i, :len(seq)] = seq
    attention_mask[i, :len(seq)] = 1

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

# One forward pass for all four responses.
with torch.inference_mode():
    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

    last_positions = attention_mask.sum(dim=1) - 1
    batch_embeddings = (
        output.last_hidden_state[
            torch.arange(len(sequences), device=device),
            last_positions,
        ]
        .float()
        .cpu()
        .contiguous()
    )

# Compare with the saved single-response embedding.
smoke = torch.load(
    OUTPUT_DIR / "smoke_test.pt",
    map_location="cpu",
    weights_only=True,
)

difference = (
    batch_embeddings[0] - smoke["embedding"]
).abs().max().item()

print("Batch embedding shape:", tuple(batch_embeddings.shape))
print("All values finite:", torch.isfinite(batch_embeddings).all().item())
print("Maximum difference from smoke test:", difference)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

assert batch_embeddings.shape == (4, 4096)
assert torch.isfinite(batch_embeddings).all()
assert difference < 0.01, "Batch output differs; investigate before scaling up."

print("Batch test passed.")

Batch embedding shape: (4, 4096)
All values finite: True
Maximum difference from smoke test: 0.25
GPU memory allocated: 14.0 GB


AssertionError: Batch output differs; investigate before scaling up.

In [ ]:
import torch
import torch.nn.functional as F

row = dataset["train"][0]
device = next(model.parameters()).device

messages = [
    {"role": "user", "content": row["prompt"]},
    {"role": "assistant", "content": row["response_1"]},
]

single_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    single_now = (
        model(input_ids=single_ids)
        .last_hidden_state[0, -1]
        .float()
        .cpu()
    )

batch_first = batch_embeddings[0].float().cpu()

smoke = torch.load(
    OUTPUT_DIR / "smoke_test.pt",
    map_location="cpu",
    weights_only=True,
)["embedding"].float()

def compare(name, a, b):
    diff = a - b
    cosine = F.cosine_similarity(a[None, :], b[None, :]).item()
    relative_l2 = (diff.norm() / (a.norm() + 1e-12)).item()

    print(f"\n{name}")
    print(f"  Max absolute difference: {diff.abs().max().item():.6f}")
    print(f"  Mean absolute difference: {diff.abs().mean().item():.6f}")
    print(f"  Relative L2 difference: {relative_l2:.8f}")
    print(f"  Cosine similarity: {cosine:.8f}")

compare("Batch vs. fresh single", batch_first, single_now)
compare("Fresh single vs. saved smoke test", single_now, smoke)


Batch vs. fresh single
  Max absolute difference: 0.250000
  Mean absolute difference: 0.016090
  Relative L2 difference: 0.00995794
  Cosine similarity: 0.99995124

Fresh single vs. saved smoke test
  Max absolute difference: 0.000000
  Mean absolute difference: 0.000000
  Relative L2 difference: 0.00000000
  Cosine similarity: 1.00000095


In [ ]:
import os
import gc
import torch
from pathlib import Path
from tqdm.auto import tqdm
from safetensors.torch import save_file, load_file

# Persistent storage on Google Drive
OUTPUT_DIR = Path("/content/drive/MyDrive/LoRe/PersonalLLM_embeddings")
CHUNK_DIR = OUTPUT_DIR / "chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 8
CHUNK_SIZE = 512       # Responses per checkpoint file
HIDDEN_SIZE = 4096
device = next(model.parameters()).device

assert model.config.hidden_size == HIDDEN_SIZE
assert torch.cuda.is_available()

pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id

model.eval()


def get_conversation(split_data, flat_idx):
    """Keep LoRe's ordering: 8 responses per prompt."""
    row_idx = flat_idx // 8
    response_idx = flat_idx % 8 + 1
    row = split_data[row_idx]

    return [
        {"role": "user", "content": str(row["prompt"] or "")},
        {
            "role": "assistant",
            "content": str(row[f"response_{response_idx}"] or ""),
        },
    ]


def embed_batch(conversations):
    """Extract the final valid token's last hidden state."""
    sequences = [
        tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            return_tensors="pt",
        )[0]
        for conv in conversations
    ]

    lengths = [len(seq) for seq in sequences]
    max_len = max(lengths)

    # Right padding, with an explicit attention mask.
    input_ids = torch.full(
        (len(sequences), max_len),
        pad_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros_like(input_ids)

    for i, seq in enumerate(sequences):
        input_ids[i, :len(seq)] = seq
        attention_mask[i, :len(seq)] = 1

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    with torch.inference_mode():
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        last_pos = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(len(sequences), device=device)

        embeddings = (
            output.last_hidden_state[batch_idx, last_pos]
            .detach()
            .cpu()
            .contiguous()
        )

    if embeddings.shape != (len(sequences), HIDDEN_SIZE):
        raise ValueError(f"Unexpected shape: {embeddings.shape}")

    if not torch.isfinite(embeddings).all():
        raise ValueError("Non-finite embedding detected.")

    return embeddings


def extract_split(split_name):
    split_data = dataset[split_name]
    n = len(split_data) * 8
    split_dir = CHUNK_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    starts = list(range(0, n, CHUNK_SIZE))
    print(f"\n{split_name}: {len(split_data):,} prompts, {n:,} responses")
    print(f"Checkpoint files: {len(starts)}")

    for chunk_id, start in enumerate(
        tqdm(starts, desc=f"{split_name} checkpoints")
    ):
        end = min(start + CHUNK_SIZE, n)
        chunk_path = split_dir / f"chunk_{chunk_id:05d}.safetensors"

        # Resume: skip only a valid, complete checkpoint.
        if chunk_path.exists():
            try:
                saved = load_file(str(chunk_path))["embeddings"]
                if saved.shape == (end - start, HIDDEN_SIZE):
                    continue
            except Exception:
                pass

            chunk_path.unlink(missing_ok=True)

        parts = []

        for batch_start in range(start, end, BATCH_SIZE):
            batch_end = min(batch_start + BATCH_SIZE, end)

            conversations = [
                get_conversation(split_data, idx)
                for idx in range(batch_start, batch_end)
            ]

            embeddings = embed_batch(conversations)
            parts.append(embeddings)

            del conversations, embeddings
            if device.type == "cuda":
                torch.cuda.empty_cache()

        chunk_embeddings = torch.cat(parts, dim=0).contiguous()
        temp_path = chunk_path.with_suffix(".tmp")

        # Write a temporary file, then rename it into place.
        save_file({"embeddings": chunk_embeddings}, str(temp_path))
        os.replace(temp_path, chunk_path)

        del parts, chunk_embeddings
        gc.collect()

    print(f"Completed checkpoints for {split_name}.")


extract_split("train")
extract_split("test")


train: 9,402 prompts, 75,216 responses
Checkpoint files: 147


train checkpoints:   0%|          | 0/147 [00:00<?, ?it/s]

Completed checkpoints for train.

test: 1,000 prompts, 8,000 responses
Checkpoint files: 16


test checkpoints:   0%|          | 0/16 [00:00<?, ?it/s]

Completed checkpoints for test.


In [ ]:
from safetensors.torch import save_file, load_file
import torch
import os
from pathlib import Path

def assemble_split(split_name):
    expected = len(dataset[split_name]) * 8
    split_dir = CHUNK_DIR / split_name
    files = sorted(split_dir.glob("chunk_*.safetensors"))

    # Allocate the final tensor and fill it in order.
    result = torch.empty(
        (expected, 4096),
        dtype=torch.bfloat16,
    )

    offset = 0
    for path in files:
        chunk = load_file(str(path))["embeddings"]
        end = offset + len(chunk)

        result[offset:end] = chunk
        offset = end

    assert offset == expected, (
        f"{split_name}: found {offset} embeddings, expected {expected}"
    )

    final_path = OUTPUT_DIR / f"{split_name}.safetensors"
    temp_path = OUTPUT_DIR / f"{split_name}.tmp"

    save_file({"embeddings": result.contiguous()}, str(temp_path))
    os.replace(temp_path, final_path)

    print(f"{final_path}")
    print(f"Shape: {tuple(result.shape)}")
    print(f"Dtype: {result.dtype}")
    print(f"Size: {final_path.stat().st_size / 1024**2:.1f} MiB")

assemble_split("train")
assemble_split("test")

/content/drive/MyDrive/LoRe/PersonalLLM_embeddings/train.safetensors
Shape: (75216, 4096)
Dtype: torch.bfloat16
Size: 587.6 MiB
/content/drive/MyDrive/LoRe/PersonalLLM_embeddings/test.safetensors
Shape: (8000, 4096)
Dtype: torch.bfloat16
Size: 62.5 MiB


# Reddit TLDR

In [ ]:
from datasets import load_dataset
from collections import Counter

reddit_ds = load_dataset(
    "openai/summarize_from_feedback",
    "comparisons",
)

print(reddit_ds)

for split_name in ["train", "validation"]:
    split = reddit_ds[split_name]
    counts = Counter(split["worker"])

    print(f"\n{split_name}:")
    print("Comparisons:", len(split))
    print("Distinct workers:", len(counts))
    print("Workers with >=50 comparisons:", sum(n >= 50 for n in counts.values()))

    row = split[0]
    print("Fields:", list(row.keys()))
    print("Prompt preview:", str(row["info"]["post"])[:200])
    print("Choice:", row["choice"])

Generating train split:   0%|          | 0/92858 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/86086 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['info', 'summaries', 'choice', 'worker', 'batch', 'split', 'extra'],
        num_rows: 92858
    })
    validation: Dataset({
        features: ['info', 'summaries', 'choice', 'worker', 'batch', 'split', 'extra'],
        num_rows: 86086
    })
})

train:
Comparisons: 92858
Distinct workers: 53
Workers with >=50 comparisons: 49
Fields: ['info', 'summaries', 'choice', 'worker', 'batch', 'split', 'extra']
Prompt preview: My boyfriend and I are long distance. We have a trip planned this summer which involves me going over to him in the USA. This will be the second time I have actually been with him in person. I am flyi
Choice: 1

validation:
Comparisons: 86086
Distinct workers: 63
Workers with >=50 comparisons: 58
Fields: ['info', 'summaries', 'choice', 'worker', 'batch', 'split', 'extra']
Prompt preview: This is my first post so please be kind :)

I know that lots of people often feel confused when they come out of a long-term relatio

In [ ]:
from collections import defaultdict
from pathlib import Path
import random
import json

# reddit_ds should already be loaded with "train" and "validation".
OUT = Path("/content/drive/MyDrive/LoRe/RedditTLDR_embeddings")
OUT.mkdir(parents=True, exist_ok=True)

# 1. Index each split by worker, preserving original row indices.
def index_by_worker(split):
    worker_rows = defaultdict(list)
    for idx, worker_id in enumerate(split["worker"]):
        worker_rows[str(worker_id)].append(idx)
    return dict(worker_rows)

train_by_worker = index_by_worker(reddit_ds["train"])
val_by_worker = index_by_worker(reddit_ds["validation"])

print("Workers in training split:", len(train_by_worker))
print("Workers in validation split:", len(val_by_worker))

# 2. Match the repository: remove the two least represented workers
#    separately from each split.
train_kept = dict(
    sorted(train_by_worker.items(), key=lambda item: len(item[1]))[2:]
)
val_kept = dict(
    sorted(val_by_worker.items(), key=lambda item: len(item[1]))[2:]
)

# 3. Split common worker IDs reproducibly.
#    Sorting first makes this deterministic across Python sessions.
common_workers = sorted(
    set(train_by_worker) & set(val_by_worker)
)

rng = random.Random(0)
rng.shuffle(common_workers)

mid = len(common_workers) // 2
seen_workers = common_workers[:mid]
unseen_workers = common_workers[mid:]

seen_set = set(seen_workers)
unseen_set = set(unseen_workers)

print("\nCommon workers:", len(common_workers))
print("Seen worker IDs:", len(seen_workers))
print("Unseen worker IDs:", len(unseen_workers))

if len(common_workers) != 40:
    print(
        "\nWARNING: The repository's paper setup reports 40 workers. "
        "This dataset version produced a different common-worker count. "
        "We'll inspect this before claiming an exact reproduction."
    )

# 4. Sample training comparisons exactly as the released script does:
#    up to 150 for seen users; up to 50 for unseen users.
#    Keep the RNG state after the worker split, as in the script.
seen_train_indices = {}
unseen_fewshot_indices = {}

for worker_id, row_indices in train_kept.items():
    if worker_id in seen_set:
        k = min(len(row_indices), 150)
        positions = rng.sample(list(set(range(len(row_indices)))), k)
        seen_train_indices[worker_id] = [
            row_indices[p] for p in positions
        ]

    elif worker_id in unseen_set:
        k = min(len(row_indices), 50)
        positions = rng.sample(list(set(range(len(row_indices)))), k)
        unseen_fewshot_indices[worker_id] = [
            row_indices[p] for p in positions
        ]

# 5. Keep all validation comparisons for selected users that remain
#    after the repository's two-worker filtering step.
selected_users = seen_set | unseen_set

val_eval_indices = {
    worker_id: row_indices
    for worker_id, row_indices in val_kept.items()
    if worker_id in selected_users
}

n_seen_pairs = sum(map(len, seen_train_indices.values()))
n_fewshot_pairs = sum(map(len, unseen_fewshot_indices.values()))
n_eval_pairs = sum(map(len, val_eval_indices.values()))

print("\n--- Extraction requirements ---")
print("Seen-user training comparisons:", f"{n_seen_pairs:,}")
print("Unseen-user few-shot comparisons:", f"{n_fewshot_pairs:,}")
print("Validation evaluation comparisons:", f"{n_eval_pairs:,}")

n_total = n_seen_pairs + n_fewshot_pairs + n_eval_pairs
print("Total selected comparisons:", f"{n_total:,}")
print("Embedding vectors before deduplication:", f"{2*n_total:,}")

# 6. Persist the exact selection to Drive.
manifest = {
    "seed": 0,
    "seen_workers": seen_workers,
    "unseen_workers": unseen_workers,
    "seen_train_indices": seen_train_indices,
    "unseen_fewshot_indices": unseen_fewshot_indices,
    "val_eval_indices": val_eval_indices,
    "counts": {
        "seen_train_pairs": n_seen_pairs,
        "unseen_fewshot_pairs": n_fewshot_pairs,
        "validation_pairs": n_eval_pairs,
    },
}

manifest_path = OUT / "selection_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved selection manifest:", manifest_path)

Workers in training split: 53
Workers in validation split: 63

Common workers: 40
Seen worker IDs: 20
Unseen worker IDs: 20

--- Extraction requirements ---
Seen-user training comparisons: 2,915
Unseen-user few-shot comparisons: 950
Validation evaluation comparisons: 71,454
Total selected comparisons: 75,319
Embedding vectors before deduplication: 150,638

Saved selection manifest: /content/drive/MyDrive/LoRe/RedditTLDR_embeddings/selection_manifest.json


In [ ]:
# Inspect all 40 selected workers against the raw and filtered splits.

seen = set(manifest["seen_workers"])
unseen = set(manifest["unseen_workers"])

train_kept_ids = set(train_kept)
val_kept_ids = set(val_kept)

print("Role | Worker | Raw train | Raw val | In filtered train | In filtered val")

for role, worker_ids in [
    ("SEEN", sorted(seen)),
    ("UNSEEN", sorted(unseen)),
]:
    for worker_id in worker_ids:
        tr = len(train_by_worker.get(worker_id, []))
        va = len(val_by_worker.get(worker_id, []))

        print(
            f"{role:6} | {worker_id} | {tr:5} | {va:5} | "
            f"{worker_id in train_kept_ids} | "
            f"{worker_id in val_kept_ids}"
        )

print("\nSelected unseen workers:", len(unseen))
print(
    "Unseen workers retained for training:",
    sum(w in train_kept_ids for w in unseen)
)

Role | Worker | Raw train | Raw val | In filtered train | In filtered val
SEEN   | 3AFaFd3w9NjDGnO51kupLyK1N44DQ2 |   300 |  2679 | True | True
SEEN   | 44Z8ttpKcY6Kr1sNymNnBA0nL0h4dZ |    84 |   485 | True | True
SEEN   | D8z53gLFLFqhZowaegbtxmSGa0jqv0 |   426 |  2408 | True | True
SEEN   | HNzkrs9geGu1YMMfZ5Qvdt0ZaCthfB |  3597 |  1138 | True | True
SEEN   | ISkqTUKflxjg9CBXeghrNug5YTZ3zS |   369 |   303 | True | True
SEEN   | Jxv4hxfb9zTVa5nsMDFlnjSX5LZ8MK |  2731 |   354 | True | True
SEEN   | KZL1qeRzHNYSfDAuOctL1iyVV8WC5N | 12985 |  5496 | True | True
SEEN   | M3icahkfAtC9CJrtKgQ7qvyZ5SD8wC |  4286 |  1881 | True | True
SEEN   | M4bdOszgybjO2qg2Dth5I1GOYAvE7V |  1494 |  2736 | True | True
SEEN   | P2p07Up4eJyvxrrVYgwtb60krbFbxI |   150 |  2082 | True | True
SEEN   | V4kZVolIcM13xFttt12rzhEn7krHK4 |   131 |    94 | True | True
SEEN   | XxIBEfyZigAJyqYeRnjaZsq4ADwPmW |   200 |    72 | True | True
SEEN   | ZzGCcAhvqF0HnKxNsUjtJFadcZdyZj |  6296 |  3510 | True | True
SEEN   | a7zXgbk

In [ ]:
import json
from pathlib import Path

OUT = Path("/content/drive/MyDrive/LoRe/RedditTLDR_embeddings")
manifest_path = OUT / "selection_manifest.json"

with open(manifest_path) as f:
    manifest = json.load(f)

bad_worker = "soP7Okyp0e1WdYxQC9IldpwnznT6V1"

# Remove the ineligible worker from the unseen-user group.
manifest["unseen_workers"] = [
    w for w in manifest["unseen_workers"]
    if w != bad_worker
]
manifest["unseen_fewshot_indices"].pop(bad_worker, None)

# Remove any validation entries for that worker.
manifest["val_eval_indices"].pop(bad_worker, None)

# Recalculate counts from the actual selected examples.
n_seen = sum(
    len(v) for v in manifest["seen_train_indices"].values()
)
n_fewshot = sum(
    len(v) for v in manifest["unseen_fewshot_indices"].values()
)
n_eval = sum(
    len(v) for v in manifest["val_eval_indices"].values()
)

manifest["counts"] = {
    "seen_train_pairs": n_seen,
    "unseen_fewshot_pairs": n_fewshot,
    "validation_pairs": n_eval,
    "seen_users": len(manifest["seen_train_indices"]),
    "unseen_users": len(manifest["unseen_fewshot_indices"]),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Updated selection manifest")
print("Seen users:", manifest["counts"]["seen_users"])
print("Unseen users:", manifest["counts"]["unseen_users"])
print("Seen training pairs:", n_seen)
print("Unseen few-shot pairs:", n_fewshot)
print("Validation pairs:", n_eval)
print("Total selected comparisons:", n_seen + n_fewshot + n_eval)
print("Saved:", manifest_path)

Updated selection manifest
Seen users: 20
Unseen users: 19
Seen training pairs: 2915
Unseen few-shot pairs: 950
Validation pairs: 71454
Total selected comparisons: 75319
Saved: /content/drive/MyDrive/LoRe/RedditTLDR_embeddings/selection_manifest.json


In [ ]:
import os
import gc
import json
import torch
from pathlib import Path
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. Load the saved selection
# ---------------------------------------------------------

REDDIT_OUT = Path(
    "/content/drive/MyDrive/LoRe/RedditTLDR_embeddings"
)
MANIFEST_PATH = REDDIT_OUT / "selection_manifest.json"

# Use a separate directory so we don't collide with any
# previous full-dataset extraction checkpoints.
CHUNK_ROOT = REDDIT_OUT / "selected_chunks"
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

# Training examples = seen-user training pairs + unseen-user few-shot pairs.
train_indices = sorted({
    int(idx)
    for mapping in [
        manifest["seen_train_indices"],
        manifest["unseen_fewshot_indices"],
    ]
    for indices in mapping.values()
    for idx in indices
})

# Validation examples = all validation comparisons for selected workers.
val_indices = sorted({
    int(idx)
    for indices in manifest["val_eval_indices"].values()
    for idx in indices
})

print("Selected training/few-shot comparisons:", len(train_indices))
print("Selected validation comparisons:", len(val_indices))
print("Total comparisons:", len(train_indices) + len(val_indices))
print("Total embedding vectors:", 2 * (len(train_indices) + len(val_indices)))

# ---------------------------------------------------------
# 2. Configure extraction
# ---------------------------------------------------------

HIDDEN_SIZE = 4096
BATCH_SIZE = 4
ROWS_PER_CHUNK = 512

device = next(model.parameters()).device
pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id

assert torch.cuda.is_available()
assert model.config.hidden_size == HIDDEN_SIZE
model.eval()

print("GPU:", torch.cuda.get_device_name(0))
print("Batch size:", BATCH_SIZE)


def get_pair(row, source_index):
    """Return the preferred/rejected responses and worker metadata."""
    choice = int(row["choice"])
    summaries = row["summaries"]

    if choice not in (0, 1):
        raise ValueError(f"Unexpected choice: {choice}")

    post = row["info"]["post"] or ""
    winner = summaries[choice]["text"] or ""
    loser = summaries[1 - choice]["text"] or ""

    return {
        "source_index": int(source_index),
        "worker": str(row["worker"]),
        "text": post,
        "winning_summary": winner,
        "losing_summary": loser,
    }


def tokenize_pair(record):
    """Use LoRe's user/post + assistant/summary chat format."""
    output = []

    for summary in [
        record["winning_summary"],
        record["losing_summary"],
    ]:
        messages = [
            {"role": "user", "content": record["text"]},
            {"role": "assistant", "content": summary},
        ]

        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_tensors="pt",
        )[0].cpu()

        output.append(ids)

    return output


def embed_pairs(records):
    """
    Embeddings are returned in interleaved order:
    winner_0, loser_0, winner_1, loser_1, ...
    """
    sequences = []

    for record in records:
        sequences.extend(tokenize_pair(record))

    n = len(sequences)
    embeddings = torch.empty(
        (n, HIDDEN_SIZE),
        dtype=torch.bfloat16,
    )

    # Length-sort batches to reduce padding.
    order = sorted(range(n), key=lambda i: len(sequences[i]))

    for start in range(0, n, BATCH_SIZE):
        positions = order[start:start + BATCH_SIZE]
        batch = [sequences[i] for i in positions]

        lengths = [len(seq) for seq in batch]
        max_len = max(lengths)

        input_ids = torch.full(
            (len(batch), max_len),
            pad_id,
            dtype=torch.long,
        )
        attention_mask = torch.zeros_like(input_ids)

        for j, seq in enumerate(batch):
            input_ids[j, :len(seq)] = seq
            attention_mask[j, :len(seq)] = 1

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        with torch.inference_mode():
            result = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            )

            last_pos = attention_mask.sum(dim=1) - 1
            batch_idx = torch.arange(len(batch), device=device)

            vectors = (
                result.last_hidden_state[batch_idx, last_pos]
                .detach()
                .to(torch.bfloat16)
                .cpu()
                .contiguous()
            )

        embeddings[torch.tensor(positions)] = vectors

        del input_ids, attention_mask, result, vectors

    del sequences
    return embeddings


# ---------------------------------------------------------
# 3. Extract a selected split with resumable checkpoints
# ---------------------------------------------------------

def extract_selected_split(split_name, data, row_indices):
    split_dir = CHUNK_ROOT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    total = len(row_indices)
    starts = list(range(0, total, ROWS_PER_CHUNK))

    print(f"\n{split_name}: {total:,} comparisons")
    print(f"Checkpoint chunks: {len(starts)}")

    for chunk_id, start in enumerate(
        tqdm(starts, desc=split_name)
    ):
        end = min(start + ROWS_PER_CHUNK, total)
        chunk_indices = row_indices[start:end]

        chunk_path = split_dir / f"chunk_{chunk_id:05d}.pt"
        temp_path = chunk_path.with_suffix(".tmp")

        # Skip only complete checkpoints with matching indices and shape.
        if chunk_path.exists():
            try:
                saved = torch.load(
                    chunk_path,
                    map_location="cpu",
                    weights_only=False,
                )

                valid = (
                    saved["source_indices"] == chunk_indices
                    and saved["embeddings"].shape
                    == (2 * len(chunk_indices), HIDDEN_SIZE)
                    and saved["worker_ids"]
                    == [
                        str(data[idx]["worker"])
                        for idx in chunk_indices
                    ]
                )
                del saved

                if valid:
                    continue
            except Exception:
                pass

            chunk_path.unlink(missing_ok=True)

        records = [
            get_pair(data[idx], idx)
            for idx in chunk_indices
        ]

        embeddings = embed_pairs(records)

        if not torch.isfinite(embeddings).all():
            raise ValueError(
                f"Non-finite embedding in {split_name}, chunk {chunk_id}"
            )

        # Keep text out of checkpoint files; only save necessary metadata.
        torch.save(
            {
                "source_indices": chunk_indices,
                "worker_ids": [r["worker"] for r in records],
                "embeddings": embeddings,
            },
            temp_path,
        )
        os.replace(temp_path, chunk_path)

        del records, embeddings
        gc.collect()
        torch.cuda.empty_cache()

    print(f"Completed {split_name}.")


# ---------------------------------------------------------
# 4. Run only the selected rows
# ---------------------------------------------------------

extract_selected_split(
    "train_selected",
    reddit_ds["train"],
    train_indices,
)

extract_selected_split(
    "validation_selected",
    reddit_ds["validation"],
    val_indices,
)

print("\nAll selected Reddit embedding checkpoints are saved.")
print("Location:", CHUNK_ROOT)

Selected training/few-shot comparisons: 3865
Selected validation comparisons: 71454
Total comparisons: 75319
Total embedding vectors: 150638
GPU: NVIDIA A100-SXM4-80GB
Batch size: 4

train_selected: 3,865 comparisons
Checkpoint chunks: 8


train_selected:   0%|          | 0/8 [00:00<?, ?it/s]

Completed train_selected.

validation_selected: 71,454 comparisons
Checkpoint chunks: 140


validation_selected:   0%|          | 0/140 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import json
from pathlib import Path

REDDIT_OUT = Path(
    "/content/drive/MyDrive/LoRe/RedditTLDR_embeddings"
)

with open(REDDIT_OUT / "selection_manifest.json") as f:
    manifest = json.load(f)

print("Seen-user training pairs:",
      sum(len(v) for v in manifest["seen_train_indices"].values()))

print("Unseen-user few-shot pairs:",
      sum(len(v) for v in manifest["unseen_fewshot_indices"].values()))

print("Validation pairs:",
      sum(len(v) for v in manifest["val_eval_indices"].values()))

Seen-user training pairs: 2915
Unseen-user few-shot pairs: 950
Validation pairs: 71454


In [ ]:
import json
import random
from pathlib import Path

REDDIT_OUT = Path(
    "/content/drive/MyDrive/LoRe/RedditTLDR_embeddings"
)
manifest_path = REDDIT_OUT / "selection_manifest.json"
backup_path = REDDIT_OUT / "selection_manifest_full_validation.json"

# Load the current manifest
with open(manifest_path) as f:
    manifest = json.load(f)

# Preserve the full selection before reducing it
if not backup_path.exists():
    with open(backup_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print("Backed up original full-validation manifest.")

# Sample at most 300 validation comparisons per worker
MAX_VAL_PER_USER = 300
rng = random.Random(42)

reduced_val = {}

for worker_id, indices in manifest["val_eval_indices"].items():
    indices = list(indices)
    n = min(MAX_VAL_PER_USER, len(indices))
    reduced_val[worker_id] = sorted(rng.sample(indices, n))

manifest["val_eval_indices"] = reduced_val
manifest["validation_per_user_cap"] = MAX_VAL_PER_USER
manifest["validation_sampling_seed"] = 42

# Update recorded counts
manifest["counts"]["validation_pairs"] = sum(
    len(v) for v in reduced_val.values()
)

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nUPDATED MANIFEST")
print("Seen-user training pairs:",
      sum(len(v) for v in manifest["seen_train_indices"].values()))
print("Unseen-user few-shot pairs:",
      sum(len(v) for v in manifest["unseen_fewshot_indices"].values()))
print("Validation workers:", len(reduced_val))
print("Validation pairs:", manifest["counts"]["validation_pairs"])
print("Total selected comparisons:",
      sum(len(v) for v in manifest["seen_train_indices"].values())
      + sum(len(v) for v in manifest["unseen_fewshot_indices"].values())
      + manifest["counts"]["validation_pairs"])
print("Saved:", manifest_path)

Backed up original full-validation manifest.

UPDATED MANIFEST
Seen-user training pairs: 2915
Unseen-user few-shot pairs: 950
Validation workers: 39
Validation pairs: 10275
Total selected comparisons: 14140
Saved: /content/drive/MyDrive/LoRe/RedditTLDR_embeddings/selection_manifest.json


In [ ]:
import json
from pathlib import Path

REDDIT_OUT = Path(
    "/content/drive/MyDrive/LoRe/RedditTLDR_embeddings"
)

# Load the updated manifest
with open(REDDIT_OUT / "selection_manifest.json") as f:
    manifest = json.load(f)

# Rebuild the reduced validation indices
val_indices = sorted({
    int(idx)
    for indices in manifest["val_eval_indices"].values()
    for idx in indices
})

print("Validation comparisons:", len(val_indices))
assert len(val_indices) == 10275

# Use a fresh directory so old full-validation chunks cannot be reused.
CHUNK_ROOT = REDDIT_OUT / "selected_chunks_eval300"
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)

# Reuse the extraction function defined earlier.
extract_selected_split(
    "validation_selected",
    reddit_ds["validation"],
    val_indices,
)

print("Reduced validation extraction completed.")

Validation comparisons: 10275

validation_selected: 10,275 comparisons
Checkpoint chunks: 21


validation_selected:   0%|          | 0/21 [00:00<?, ?it/s]

Completed validation_selected.
Reduced validation extraction completed.


In [ ]:
import os
import gc
import json
import pickle
import torch
from pathlib import Path
from tqdm.auto import tqdm

REDDIT_OUT = Path(
    "/content/drive/MyDrive/LoRe/RedditTLDR_embeddings"
)

# The two extraction runs used different checkpoint directories.
TRAIN_CHUNKS = REDDIT_OUT / "selected_chunks" / "train_selected"
VAL_CHUNKS = (
    REDDIT_OUT / "selected_chunks_eval300" / "validation_selected"
)

with open(REDDIT_OUT / "selection_manifest.json") as f:
    manifest = json.load(f)

train_data = reddit_ds["train"]
val_data = reddit_ds["validation"]
HIDDEN_SIZE = 4096

# Expected source rows from the saved manifest.
expected_train = {
    int(idx)
    for mapping in [
        manifest["seen_train_indices"],
        manifest["unseen_fewshot_indices"],
    ]
    for indices in mapping.values()
    for idx in indices
}

expected_val = {
    int(idx)
    for indices in manifest["val_eval_indices"].values()
    for idx in indices
}

assert len(expected_train) == 3865, len(expected_train)
assert len(expected_val) == 10275, len(expected_val)


def assemble_split(chunk_dir, dataset_split, expected_indices, output_name):
    if not chunk_dir.exists():
        raise FileNotFoundError(f"Missing directory: {chunk_dir}")

    chunk_files = sorted(chunk_dir.glob("chunk_*.pt"))
    if not chunk_files:
        raise FileNotFoundError(f"No checkpoints found in {chunk_dir}")

    data = train_data if dataset_split == "train" else val_data

    worker_results = {}
    processed = set()

    for path in tqdm(chunk_files, desc=f"Assembling {dataset_split}"):
        saved = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        indices = [int(i) for i in saved["source_indices"]]
        worker_ids = [str(w) for w in saved["worker_ids"]]
        embeddings = saved["embeddings"]

        assert embeddings.shape == (2 * len(indices), HIDDEN_SIZE), (
            path, embeddings.shape
        )
        assert len(worker_ids) == len(indices)

        for j, source_idx in enumerate(indices):
            if source_idx in processed:
                raise ValueError(f"Duplicate source index: {source_idx}")

            processed.add(source_idx)
            row = data[source_idx]

            worker_id = str(row["worker"])
            if worker_id != worker_ids[j]:
                raise ValueError(f"Worker mismatch at index {source_idx}")

            choice = int(row["choice"])
            summaries = row["summaries"]

            if choice not in (0, 1):
                raise ValueError(f"Invalid choice at index {source_idx}")

            entry = {
                "text": row["info"]["post"] or "",
                "winning_summary": summaries[choice]["text"] or "",
                "losing_summary": summaries[1 - choice]["text"] or "",
                "embeddings": {
                    "winning": [embeddings[2 * j].clone()],
                    "losing": [embeddings[2 * j + 1].clone()],
                },
            }

            worker_results.setdefault(worker_id, []).append(entry)

        del saved, embeddings
        gc.collect()

    # Verify that the checkpoint files exactly match the manifest.
    if processed != expected_indices:
        missing = expected_indices - processed
        extra = processed - expected_indices
        raise ValueError(
            f"Selection mismatch: {len(missing)} missing, "
            f"{len(extra)} extra indices."
        )

    # Match the repository's ascending-by-worker-count ordering.
    worker_results = dict(
        sorted(worker_results.items(), key=lambda item: len(item[1]))
    )

    output_path = REDDIT_OUT / output_name
    temp_path = output_path.with_suffix(".tmp")

    with open(temp_path, "wb") as f:
        pickle.dump(worker_results, f, protocol=pickle.HIGHEST_PROTOCOL)

    os.replace(temp_path, output_path)

    total_pairs = sum(len(v) for v in worker_results.values())

    print(f"\nSaved: {output_path}")
    print(f"Comparisons: {total_pairs:,}")
    print(f"Workers: {len(worker_results)}")
    print(f"File size: {output_path.stat().st_size / 1024**2:.1f} MiB")

    del worker_results
    gc.collect()

    return total_pairs


n_train = assemble_split(
    TRAIN_CHUNKS,
    "train",
    expected_train,
    "tldr_embeddings_train.pkl",
)

n_val = assemble_split(
    VAL_CHUNKS,
    "validation",
    expected_val,
    "tldr_embeddings_val.pkl",
)

# Preserve the exact user split for the later training script.
with open(REDDIT_OUT / "user_split.json", "w") as f:
    json.dump(
        {
            "seen_workers": manifest["seen_workers"],
            "unseen_workers": manifest["unseen_workers"],
            "seen_train_indices": manifest["seen_train_indices"],
            "unseen_fewshot_indices": manifest["unseen_fewshot_indices"],
            "validation_sampling_seed": manifest.get(
                "validation_sampling_seed", 42
            ),
        },
        f,
        indent=2,
    )

assert n_train == 3865
assert n_val == 10275

print("\nSUCCESS: Reddit embedding preparation complete.")
print("Training/few-shot pairs:", n_train)
print("Held-out validation pairs:", n_val)
print("Total pairs:", n_train + n_val)
print("Total embedding vectors:", 2 * (n_train + n_val))
print("Output directory:", REDDIT_OUT)

Assembling train:   0%|          | 0/8 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/LoRe/RedditTLDR_embeddings/tldr_embeddings_train.pkl
Comparisons: 3,865
Workers: 39
File size: 68.5 MiB


Assembling validation:   0%|          | 0/21 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/LoRe/RedditTLDR_embeddings/tldr_embeddings_val.pkl
Comparisons: 10,275
Workers: 39
File size: 181.6 MiB

SUCCESS: Reddit embedding preparation complete.
Training/few-shot pairs: 3865
Held-out validation pairs: 10275
Total pairs: 14140
Total embedding vectors: 28280
Output directory: /content/drive/MyDrive/LoRe/RedditTLDR_embeddings
